# Hammurab.AI — Qwen2.5-7B + Unsloth QLoRA (Free Colab T4)
**CIF425 Term Project**

## ⚠️ KULLANIM

**Hücreleri SIRAYLA çalıştır, atlamadan!** Her bölümün sonunda kontrol var, eksik adım varsa hata mesajı seni doğru yere yönlendirir.

1. **Runtime > Change runtime type > T4 GPU** seç
2. Cell 1 → 14'ü sırayla çalıştır (Cell 14 = fine-tune, ~1.5-2 saat)
3. Bittiğinde Cell 17+ (test) çalıştır

**Free Colab kısıtları:**
- Max 12 saat / 90 dk idle disconnect
- Tab kapanırsa eğitim ölür → sekmeyi açık tut!
- Checkpoint'ler Drive'a yazılır, disconnect olursa Cell 16 ile devam edilebilir.

## Adım 1 — Repo Clone

In [ ]:
!git clone https://github.com/Alp33er/hammurab.ai.git
%cd hammurab.ai
!git checkout development
!git pull origin development

## Adım 2 — Bağımlılıkları Kur (3-5 dk)

Unsloth + TRL + PEFT + bitsandbytes paketleri kurulur.

In [ ]:
!pip install -q -r training/requirements.txt

# Kurulum doğrulama — eğer ImportError görürsen kurulum başarısız
import importlib, sys
for pkg in ["unsloth", "trl", "peft", "bitsandbytes", "transformers", "datasets"]:
    try:
        importlib.import_module(pkg)
        print(f"  ✓ {pkg}")
    except ImportError as e:
        print(f"  ✗ {pkg} — KURULAMADI: {e}")
        sys.exit(1)
print("\n✅ Tüm paketler hazır.")

## Adım 3 — GPU Kontrol

In [ ]:
import torch
assert torch.cuda.is_available(), "❌ GPU YOK — Runtime > Change runtime type > T4 GPU seç!"
print(f"GPU:  {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"BF16: {torch.cuda.is_bf16_supported()} (T4'te False normaldir)")

## Adım 4 — Google Drive'ı Bağla

Checkpoint'ler Drive'a yazılır → disconnect olsa bile kayıp az.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os
OUTPUT_DIR = "/content/drive/MyDrive/hammurab_lora"
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.environ["HAMMURAB_OUTPUT_DIR"] = OUTPUT_DIR
print(f"✅ Output dizini: {OUTPUT_DIR}")

## Adım 5 — Keep-Alive

Browser tab'ını **AÇIK** tut. Bu hücre her 60 sn'de tıklama simüle eder.

In [ ]:
from IPython.display import display, Javascript
display(Javascript('''
function ClickConnect(){
    console.log("Keep-alive: " + new Date().toLocaleString());
    document.querySelector("colab-toolbar-button#connect")?.click();
}
setInterval(ClickConnect, 60000);
'''))
print("✅ Keep-alive aktif. Tab'ı KAPATMA.")

## Adım 6 — Veri Seti Oluştur

Kanun JSON'larından Qwen ChatML formatında ~18K Q&A çifti üretir.

In [ ]:
!python training/prepare_dataset.py

# Doğrulama
import json, os
data_path = "training/data/hukuk_qa.jsonl"
assert os.path.exists(data_path), f"❌ Veri seti üretilemedi: {data_path}"
with open(data_path) as f:
    sample = json.loads(f.readline())
assert "messages" in sample, (
    "❌ Veri seti eski formatta! GitHub repo eski olabilir.\n"
    "   Çöz: !git pull origin development sonra Adım 6'yı tekrar çalıştır."
)
print("\n✅ Örnek (yeni format):")
for msg in sample["messages"]:
    print(f"\n[{msg['role'].upper()}]")
    print(msg["content"][:300])

## Adım 7 — Fine-Tune Başlat (~1.5-2 saat T4'te)

**Konfigürasyon:**
- Qwen2.5-7B-Instruct + 4-bit + LoRA r=16
- max_seq=1024, batch=2, grad_accum=4 → eff. batch=8
- Her 200 adımda Drive'a checkpoint

**Daha fazla epoch için:** Aşağıdaki override satırını yorum-dışına çıkar.

**ÖNEMLİ:** Bu hücre çalışırken sekmeyi kapatma, başka tab'a geçme sorun değil.

In [ ]:
# İsteğe bağlı override (yorum kaldır):
# os.environ["HAMMURAB_EPOCHS"] = "3"
# os.environ["HAMMURAB_MAX_SEQ"] = "2048"

!python training/fine_tune.py

# Doğrulama — adapter dosyaları üretildi mi?
import os
adapter_cfg = os.path.join(OUTPUT_DIR, "adapter_config.json")
if os.path.exists(adapter_cfg):
    print(f"\n✅ Eğitim tamam! Adapter: {OUTPUT_DIR}")
    !ls -lh "$OUTPUT_DIR"
else:
    print(
        f"\n⚠️  adapter_config.json bulunamadı: {OUTPUT_DIR}\n"
        f"   Eğitim yarıda mı kaldı? Logları kontrol et."
    )

## Adım 7b — Disconnect Olduysa: Checkpoint'ten Devam Et

Eğitim ortasında session koptuysa: yeni runtime aç, **Adım 1-5**'i tekrarla, sonra **bu hücreyi** çalıştır (Adım 7'yi değil).

In [ ]:
# Yorum kaldır:
# os.environ["HAMMURAB_RESUME"] = "1"
# !python training/fine_tune.py

## Adım 8 — Modeli Test Et

**Önemli:** Önce Adım 7 tamamlanmış olmalı. Adapter yoksa bu hücre seni Adım 7'ye yönlendirir.

In [ ]:
import os

# Pre-check: adapter var mı?
adapter_cfg = os.path.join(OUTPUT_DIR, "adapter_config.json")
assert os.path.exists(adapter_cfg), (
    f"❌ LoRA adapter bulunamadı: {adapter_cfg}\n"
    f"   Önce Adım 7'yi (Fine-tune) çalıştırıp tamamlanmasını bekle."
)

from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=OUTPUT_DIR,
    max_seq_length=1024,
    dtype=None,
    load_in_4bit=True,
)
tokenizer = get_chat_template(tokenizer, chat_template="qwen-2.5")
FastLanguageModel.for_inference(model)

SYSTEM = (
    "Sen Türk hukuku konusunda uzman bir hukuk araştırma asistanısın. "
    "Soruları verilen mevzuat metinlerine dayanarak yanıtla, "
    "ilgili kanun maddesini ve referansını mutlaka göster, "
    "context'te olmayan bilgiyi uydurma."
)

def sor(soru, mevzuat=""):
    user_content = soru
    if mevzuat:
        user_content += f"\n\n[İlgili Mevzuat]\n{mevzuat}"
    messages = [
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content": user_content},
    ]
    inputs = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
    ).to("cuda")
    outputs = model.generate(
        inputs, max_new_tokens=512, do_sample=True,
        temperature=0.7, top_p=0.9, repetition_penalty=1.1,
    )
    return tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True).strip()

print("✅ Model hazır. Aşağıdaki hücrelerle test et.")

In [ ]:
print(sor("İş kazası durumunda işçinin hakları nelerdir?"))

In [ ]:
print(sor("Türk Borçlar Kanunu madde 49 ne diyor?"))

In [ ]:
print(sor("Kıdem tazminatı nasıl hesaplanır?"))

## Adım 9 — Modeli Lokal'e İndir

**LoRA adapter** sadece ~50-100 MB. Lokal `app.py` Qwen2.5-7B-Instruct'ı kendi indirir, üzerine bu adapter'ı yükler.

In [ ]:
!cd "$OUTPUT_DIR" && zip -r /content/hammurab_lora.zip ./*
!ls -lh /content/hammurab_lora.zip

from google.colab import files
files.download("/content/hammurab_lora.zip")

## Adım 10 — (Opsiyonel) HuggingFace Hub'a Push

HF'e yüklersen `app.py`'da `from_pretrained("USERNAME/hammurab-qwen-7b-lora")` ile direkt çekersin.

In [ ]:
# from huggingface_hub import login
# login()
# model.push_to_hub("USERNAME/hammurab-qwen-7b-lora")
# tokenizer.push_to_hub("USERNAME/hammurab-qwen-7b-lora")